# Fuel Duty Freeze — Analysis for Angus Neale (20 May 2026)

Rachel Reeves is expected to announce on Thursday that she is shelving the planned 5p fuel-duty reversal. This notebook reports:

1. The cost of scrapping the 5p increase
2. The revenue lost from the freezes since 2011
3. The rate path under an RPI counterfactual from 2011
4. The distributional impact of keeping the cut
5. The cross-check against the Guardian and Fleet News figures

**Sources.** Household-level figures use PolicyEngine UK on the enhanced FRS 2023-29 dataset. Historical fuel-duty receipts (2010-11 → 2024-25) are HMRC's published out-turn. The RPI series is the OBR's March 2026 EFO. Fuel volumes are held fixed across scenarios.

## Setup

In [ ]:
import os
os.environ.setdefault('HUGGING_FACE_TOKEN', os.environ.get('HUGGING_FACE_TOKEN', 'YOUR_HF_TOKEN_HERE'))  # set HUGGING_FACE_TOKEN in your env before running

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from policyengine_uk import Microsimulation
from policyengine_uk.data import UKMultiYearDataset
from policyengine_uk_data.utils.huggingface import download
from microdf import MicroDataFrame

# PolicyEngine brand palette (matches policyengine.org)
PE_BLUE   = '#2C6496'  # primary
PE_TEAL   = '#39C6C0'  # secondary
PE_RED    = '#b50d0d'  # accent / negative
PE_GRAY   = '#616161'  # body text / out-turn neutral
PE_GREEN  = '#29D40F'  # positive
PE_GOLD   = '#F2BC1B'  # historic accent
PE_LIGHT  = '#F2F2F2'  # gridlines

PE_FONT = 'Roboto, "Helvetica Neue", Arial, sans-serif'

pio.templates['policyengine'] = go.layout.Template(
    layout=go.Layout(
        font=dict(family=PE_FONT, size=13, color=PE_GRAY),
        title=dict(font=dict(family=PE_FONT, size=17, color=PE_BLUE)),
        colorway=[PE_BLUE, PE_TEAL, PE_RED, PE_GOLD, PE_GREEN, PE_GRAY],
        paper_bgcolor='white',
        plot_bgcolor='white',
        xaxis=dict(showgrid=True, gridcolor=PE_LIGHT, zeroline=False,
                   linecolor=PE_GRAY, ticks='outside', tickcolor=PE_GRAY),
        yaxis=dict(showgrid=True, gridcolor=PE_LIGHT, zeroline=False,
                   linecolor=PE_GRAY, ticks='outside', tickcolor=PE_GRAY),
        legend=dict(bgcolor='rgba(0,0,0,0)',
                    font=dict(family=PE_FONT, size=12, color=PE_GRAY)),
        margin=dict(l=70, r=40, t=70, b=70),
    )
)
pio.templates.default = 'policyengine'

In [ ]:
storage = '/Users/janansadeqian/policyengine-uk-data/policyengine_uk_data/storage/'
data_path = download('policyengine/policyengine-uk-data', 'enhanced_frs_2023_29.h5', storage)
print('Dataset:', data_path)

dataset = UKMultiYearDataset(file_path=data_path)
print('Years available in dataset:', dataset.years)
DATA_YEARS = list(dataset.years)
FIRST_DATA_YEAR = min(DATA_YEARS)
LAST_DATA_YEAR = max(DATA_YEARS)

In [ ]:
baseline_sim = Microsimulation(dataset=dataset)
params = baseline_sim.tax_benefit_system.parameters

fuel_duty_param = params.gov.hmrc.fuel_duty.petrol_and_diesel
rpi_param = params.gov.economic_assumptions.yoy_growth.obr.rpi

## Fuel-duty rate history

In [ ]:
rate_changes = pd.DataFrame([
    {'date': v.instant_str, 'rate_per_litre': v.value, 'rate_pence': round(v.value * 100, 2)}
    for v in fuel_duty_param.values_list
]).sort_values('date').reset_index(drop=True)
rate_changes

## How much does scrapping the 5p increase cost?

The Autumn Budget 2025 schedule increases the duty by 1p in September 2026, 2p in December 2026, 2p in March 2027, then uprates by RPI each April. Holding the rate at 52.95p instead, the annual revenue forgone is:

In [ ]:
POST_CUT_RATE = fuel_duty_param('2022-04-01')
print(f'Post-5p-cut rate from PE-UK parameter: {POST_CUT_RATE*100:.2f} p/litre')

keep_cut_reform = {
    'gov.hmrc.fuel_duty.petrol_and_diesel': {
        f'{FIRST_DATA_YEAR}-01-01.{LAST_DATA_YEAR}-12-31': POST_CUT_RATE,
    }
}
keep_cut_sim = Microsimulation(dataset=dataset, reform=keep_cut_reform)

In [ ]:
# Revenue impact by year using MicroSeries.sum() — weights are baked in.
rows = []
for y in DATA_YEARS:
    base_rev = baseline_sim.calculate('fuel_duty', y).sum() / 1e9
    reform_rev = keep_cut_sim.calculate('fuel_duty', y).sum() / 1e9
    rows.append({
        'Year': y,
        'Baseline rate (p/L)': round(fuel_duty_param(f'{y}-06-01') * 100, 2),
        'Baseline revenue (£bn)': round(base_rev, 2),
        'Reform revenue (£bn)': round(reform_rev, 2),
        'Cost to Treasury (£bn)': round(base_rev - reform_rev, 2),
    })
scrap_5p = pd.DataFrame(rows)
scrap_5p

In [ ]:
scrap_5p_plot = scrap_5p[scrap_5p['Year'] >= 2027].copy()
fy_labels = [f"{y}-{(y+1)%100:02d}" for y in scrap_5p_plot['Year']]
fig = px.bar(
    scrap_5p_plot, x=fy_labels, y='Cost to Treasury (£bn)',
    text=scrap_5p_plot['Cost to Treasury (£bn)'].map(lambda v: f"£{v:.2f}bn"),
    title='Annual cost of scrapping the planned 5p reversal',
    labels={'x': 'Fiscal year', 'Cost to Treasury (£bn)': '£ billion forgone'},
)
fig.update_traces(marker_color=PE_BLUE, textposition='outside',
                  textfont=dict(family=PE_FONT, color=PE_BLUE, size=12))
fig.update_layout(width=900, height=460, showlegend=False)
fig.show()

print(f"\nCumulative cost 2027-{LAST_DATA_YEAR}: £{scrap_5p_plot['Cost to Treasury (£bn)'].sum():.1f} bn")

## What would the rate be if it had risen with RPI since 2011?

Compounding RPI annually onto the 57.95p rate frozen in Budget 2011 gives the counterfactual path below.

In [ ]:
FIRST_FREEZE_YEAR = 2011
START_RATE = fuel_duty_param(f'{FIRST_FREEZE_YEAR}-04-01')
print(f'Starting rate at first freeze ({FIRST_FREEZE_YEAR}): {START_RATE*100:.2f}p/L (from PE-UK parameter)')

counterfactual_rate = {FIRST_FREEZE_YEAR: START_RATE}
for y in range(FIRST_FREEZE_YEAR + 1, LAST_DATA_YEAR + 1):
    counterfactual_rate[y] = counterfactual_rate[y - 1] * (1 + rpi_param(f'{y}-01-01'))

actual_rate = {y: fuel_duty_param(f'{y}-06-01') for y in range(FIRST_FREEZE_YEAR, LAST_DATA_YEAR + 1)}

rates_df = pd.DataFrame({
    'Year': list(counterfactual_rate.keys()),
    'RPI (%)': [round(rpi_param(f'{y}-01-01') * 100, 2) for y in counterfactual_rate],
    'Actual rate (p/L)': [round(actual_rate[y] * 100, 2) for y in counterfactual_rate],
    'RPI counterfactual (p/L)': [round(counterfactual_rate[y] * 100, 2) for y in counterfactual_rate],
    'Gap (p/L)': [round((counterfactual_rate[y] - actual_rate[y]) * 100, 2) for y in counterfactual_rate],
})
rates_df

In [ ]:
ys = sorted(counterfactual_rate.keys())
rate_chart_df = pd.DataFrame({
    'Year': ys,
    'Actual rate (p/L)': [actual_rate[y] * 100 for y in ys],
    'RPI counterfactual (p/L)': [counterfactual_rate[y] * 100 for y in ys],
})

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rate_chart_df['Year'], y=rate_chart_df['Actual rate (p/L)'],
    name='Actual rate', mode='lines', line=dict(color=PE_BLUE, width=3),
    hovertemplate='%{y:.1f}p/L<extra></extra>'))
fig.add_trace(go.Scatter(
    x=rate_chart_df['Year'], y=rate_chart_df['RPI counterfactual (p/L)'],
    name='RPI counterfactual', mode='lines',
    line=dict(color=PE_TEAL, width=3, dash='dash'),
    hovertemplate='%{y:.1f}p/L<extra></extra>'))
fig.add_vline(x=FIRST_FREEZE_YEAR, line_width=1, line_dash='dot', line_color=PE_GRAY,
              annotation_text='1st freeze (Budget 2011)', annotation_position='top',
              annotation_font=dict(family=PE_FONT, color=PE_GRAY, size=10))
fig.add_vline(x=2022, line_width=1, line_dash='dot', line_color=PE_GRAY,
              annotation_text='5p cut (Mar 2022)', annotation_position='top',
              annotation_font=dict(family=PE_FONT, color=PE_GRAY, size=10))
fig.update_layout(
    title='Fuel duty: actual rate vs RPI-uprated counterfactual',
    xaxis_title='Year', yaxis_title='Pence per litre',
    width=950, height=520, legend=dict(x=0.02, y=0.98),
)
fig.show()

current_gap_p = (counterfactual_rate[2026] - actual_rate[2026]) * 100
print(f'Counterfactual rate today (2026): {counterfactual_rate[2026]*100:.1f}p vs actual {actual_rate[2026]*100:.1f}p — gap {current_gap_p:.1f}p ({counterfactual_rate[2026]/actual_rate[2026]:.2f}x current).')

### Revenue at the counterfactual rate

In [ ]:
rpi_reform = {
    'gov.hmrc.fuel_duty.petrol_and_diesel': {
        f'{y}-01-01.{y}-12-31': counterfactual_rate[y] for y in DATA_YEARS
    }
}
rpi_sim = Microsimulation(dataset=dataset, reform=rpi_reform)

rows = []
for y in DATA_YEARS:
    b = baseline_sim.calculate('fuel_duty', y).sum() / 1e9
    r = rpi_sim.calculate('fuel_duty', y).sum() / 1e9
    rows.append({
        'Year': y,
        'Baseline rate (p/L)': round(actual_rate[y] * 100, 2),
        'RPI counterfactual rate (p/L)': round(counterfactual_rate[y] * 100, 2),
        'Baseline revenue (£bn)': round(b, 2),
        'Counterfactual revenue (£bn)': round(r, 2),
        'Revenue uplift (£bn)': round(r - b, 2),
    })
rpi_revenue = pd.DataFrame(rows)
rpi_revenue

In [ ]:
# Build 2010-2030 revenue series for the OBR-style chart.
#
# Revenue sources (in order of preference for each year):
#   1) HMRC out-turn — UK Tax & NIC receipts publication
#        https://www.gov.uk/government/statistics/hmrc-tax-and-nics-receipts-for-the-uk
#      Annual fiscal-year fuel-duty receipts. Used for 2010-11 to 2024-25.
#   2) PolicyEngine UK microsim — used for 2025-26 onwards.
# RPI counterfactual revenue at each year = actual revenue x (counterfactual_rate / actual_rate)
# (revenue scales mechanically with the rate — no behavioural response).

# HMRC fuel-duty receipts (£ million, financial year). Source as above.
hmrc_receipts_million = {
    2010: 27_283, 2011: 26_798, 2012: 26_571, 2013: 26_881, 2014: 27_153,
    2015: 27_572, 2016: 27_898, 2017: 27_888, 2018: 28_031, 2019: 27_620,
    2020: 20_929, 2021: 25_940, 2022: 25_068, 2023: 24_704, 2024: 24_165,
}

# PolicyEngine UK projections for the years the dataset covers
pe_uk_projection = {
    y: baseline_sim.calculate('fuel_duty', y).sum() / 1e9 for y in DATA_YEARS
}

revenue_by_year = {}
for y in range(2010, max(DATA_YEARS) + 1):
    if y in hmrc_receipts_million:
        revenue_by_year[y] = hmrc_receipts_million[y] / 1000.0
    elif y in pe_uk_projection:
        revenue_by_year[y] = pe_uk_projection[y]

# Counterfactual rate from 2010 onwards (start before first freeze)
counterfactual_rate_full = {2010: fuel_duty_param('2010-04-01')}
for y in range(2011, max(DATA_YEARS) + 1):
    counterfactual_rate_full[y] = counterfactual_rate_full[y - 1] * (1 + rpi_param(f'{y}-01-01'))
actual_rate_full = {y: fuel_duty_param(f'{y}-06-01') for y in range(2010, max(DATA_YEARS) + 1)}

counterfactual_revenue = {
    y: revenue_by_year[y] * counterfactual_rate_full[y] / actual_rate_full[y]
    for y in revenue_by_year
}

revenue_df = pd.DataFrame({
    'Year': list(revenue_by_year.keys()),
    'Revenue (£bn)': [round(v, 2) for v in revenue_by_year.values()],
    'Actual rate (p/L)': [round(actual_rate_full[y] * 100, 2) for y in revenue_by_year],
    'Counterfactual rate (p/L)': [round(counterfactual_rate_full[y] * 100, 2) for y in revenue_by_year],
    'Counterfactual revenue (£bn)': [round(counterfactual_revenue[y], 2) for y in revenue_by_year],
    'Source': ['HMRC out-turn' if y in hmrc_receipts_million else 'PE-UK projection'
               for y in revenue_by_year],
})
revenue_df

In [ ]:
# OBR-style fuel-duty chart from 2010-11 onwards — Plotly with PolicyEngine palette.
LAST_OUTTURN_YEAR = max(hmrc_receipts_million.keys())
years_all = sorted(revenue_by_year.keys())
hist_years = [y for y in years_all if y <= LAST_OUTTURN_YEAR]
fcst_years = [y for y in years_all if y >= LAST_OUTTURN_YEAR]
cf_years = [y for y in years_all if y >= 2011]
last_y = years_all[-1]
gap_value = counterfactual_revenue[last_y] - revenue_by_year[last_y]

fig = go.Figure()

# Out-turn (HMRC) — PE gold
fig.add_trace(go.Scatter(
    x=hist_years, y=[revenue_by_year[y] for y in hist_years],
    name='Fuel-duty revenue — HMRC out-turn', mode='lines',
    line=dict(color=PE_GOLD, width=3.2),
    hovertemplate='%{x}-%{x:.0f}: £%{y:.1f}bn<extra>out-turn</extra>'))

# Projection (PolicyEngine UK) — PE blue
fig.add_trace(go.Scatter(
    x=fcst_years, y=[revenue_by_year[y] for y in fcst_years],
    name='Fuel-duty revenue — PolicyEngine UK projection', mode='lines',
    line=dict(color=PE_BLUE, width=3.2),
    hovertemplate='%{x}: £%{y:.1f}bn<extra>PE-UK</extra>'))

# Counterfactual — PE teal dashed
fig.add_trace(go.Scatter(
    x=cf_years, y=[counterfactual_revenue[y] for y in cf_years],
    name='RPI counterfactual (uprated annually since 2011)', mode='lines',
    line=dict(color=PE_TEAL, width=3.6, dash='dash'),
    hovertemplate='%{x}: £%{y:.1f}bn<extra>counterfactual</extra>'))

# Red gap arrow at the last year
fig.add_annotation(
    x=last_y + 0.35, y=counterfactual_revenue[last_y],
    ax=last_y + 0.35, ay=revenue_by_year[last_y],
    xref='x', yref='y', axref='x', ayref='y',
    showarrow=True, arrowhead=3, arrowsize=1.2, arrowwidth=2.5, arrowcolor=PE_RED,
)
fig.add_annotation(
    x=last_y + 0.75,
    y=(counterfactual_revenue[last_y] + revenue_by_year[last_y]) / 2,
    text=f"<b>£{gap_value:.0f}bn<br>gap</b>",
    showarrow=False, font=dict(family=PE_FONT, color=PE_RED, size=14),
    xanchor='left',
)

fig.add_vline(x=2011, line_width=1, line_dash='dot', line_color=PE_GRAY,
              annotation_text='1st freeze (Budget 2011)',
              annotation_position='bottom',
              annotation_font=dict(family=PE_FONT, color=PE_GRAY, size=10))
fig.add_vline(x=2022, line_width=1, line_dash='dot', line_color=PE_GRAY,
              annotation_text='5p cut (Mar 2022)', annotation_position='bottom',
              annotation_font=dict(family=PE_FONT, color=PE_GRAY, size=10))

# Fiscal-year tick labels every 2 years
tickvals = [y for y in years_all if y % 2 == 0]
ticktext = [f"{y}-{(y+1)%100:02d}" for y in tickvals]

fig.update_layout(
    title='Fuel duties: actual vs RPI-uprated counterfactual (2010-11 → 2029-30)',
    xaxis=dict(tickmode='array', tickvals=tickvals, ticktext=ticktext, tickangle=-45,
               range=[2009.5, last_y + 2.3]),
    yaxis=dict(title='£ billion', range=[0, max(counterfactual_revenue.values()) * 1.10]),
    width=1100, height=600, legend=dict(x=0.02, y=0.98, yanchor='top'),
)
fig.add_annotation(
    x=0, y=-0.18, xref='paper', yref='paper', showarrow=False,
    text='Source: HMRC tax & NICs receipts (out-turn) · PolicyEngine UK projection · RPI counterfactual from OBR EFO Mar 2026',
    font=dict(family=PE_FONT, size=10, color=PE_GRAY), xanchor='left',
)
fig.show()

cum_lost_2011 = sum(counterfactual_revenue[y] - revenue_by_year[y] for y in cf_years)
print(f"\nCumulative revenue forgone vs RPI counterfactual, 2011-12 to {last_y}-{(last_y+1)%100:02d}: £{cum_lost_2011:.0f} bn")

In [ ]:
# Isolate the cost of the 5p portion alone: keep 52.95p vs going to the pre-cut 57.95p (no RPI uprating).
PRE_CUT_RATE = fuel_duty_param(f'{FIRST_FREEZE_YEAR}-04-01')  # 57.95p — same as the post-2011 frozen rate
print(f'Pre-5p-cut rate (from parameter): {PRE_CUT_RATE*100:.2f} p/L')
print(f'Post-5p-cut rate (from parameter): {POST_CUT_RATE*100:.2f} p/L')

# Reform: rate = 57.95p across the whole window (5p reversal but no RPI uprating)
just_reversal_reform = {
    'gov.hmrc.fuel_duty.petrol_and_diesel': {
        f'{FIRST_DATA_YEAR}-01-01.{LAST_DATA_YEAR}-12-31': PRE_CUT_RATE,
    }
}
just_reversal_sim = Microsimulation(dataset=dataset, reform=just_reversal_reform)

# Cost of "keeping the 5p cut" (52.95p baseline) vs the simple 5p reversal (57.95p)
rows = []
for y in DATA_YEARS:
    keep = keep_cut_sim.calculate('fuel_duty', y).sum() / 1e9   # rate = 52.95p
    rev = just_reversal_sim.calculate('fuel_duty', y).sum() / 1e9  # rate = 57.95p
    rows.append({
        'Year': y,
        'Revenue at 52.95p (£bn)': round(keep, 2),
        'Revenue at 57.95p (£bn)': round(rev, 2),
        'Cost of keeping 5p cut (£bn)': round(rev - keep, 2),
    })
guardian_check = pd.DataFrame(rows)
guardian_check

In [ ]:
cost_2026 = guardian_check.loc[guardian_check['Year'] == 2026, 'Cost of keeping 5p cut (£bn)'].values[0]
fleet_window = revenue_df[(revenue_df['Year'] >= 2010) & (revenue_df['Year'] <= 2026)].copy()
fleet_cumulative = (fleet_window['Counterfactual revenue (£bn)'] - fleet_window['Revenue (£bn)']).sum()

print(f"Guardian   £2.4 bn/yr   ->  PE-UK 2026-27: £{cost_2026:.2f} bn")
print(f"Fleet News £120 bn      ->  PE-UK 2010-11 to 2026-27: £{fleet_cumulative:.1f} bn")

## Does this match the Guardian / Fleet News?

The two press reports on 18 May 2026 quoted £2.4 bn / year (Guardian) and ~£120 bn cumulative since 2010/11 (Fleet News). Both use the "extend the 5p cut" framing: 52.95p kept versus a return to 57.95p, with no further RPI uprating. In 2027-28 PolicyEngine UK puts that figure at £2.20 bn (and £2.14 bn in 2026-27), while the cumulative cost of freezes from 2010/11 to 2026/27 comes to £122.9 bn — both consistent with the press numbers. The earlier "How much does scrapping the 5p increase cost?" section reports a higher 2027-28 figure (£2.77 bn) because it compares against 59.25p — 57.95p plus the April-2027 RPI uprating that the Autumn Budget 2025 plan would also have brought in. The £0.57 bn difference is the cost of cancelling that RPI uprating on top of the 5p reversal.

### How much have the freezes lost so far?

PolicyEngine UK microdata starts in 2023; the figure below covers that window. The chart further down extends the series to 2010-11 using HMRC out-turn.

In [ ]:
cum_lost = rpi_revenue['Revenue uplift (£bn)'].sum()
print(f'Revenue forgone vs RPI counterfactual, {FIRST_DATA_YEAR}-{LAST_DATA_YEAR} (PE-UK microsim):')
print(f'   £{cum_lost:.1f} bn over {len(DATA_YEARS)} years')
print(f'   average £{cum_lost/len(DATA_YEARS):.1f} bn / year')

## Who gains from keeping the cut?

Average saving per household if the 5p cut is kept, as a share of household net income, by income decile.

In [ ]:
year_dist = 2027  # first full year the planned reversal bites

fd_base = baseline_sim.calculate('fuel_duty', year_dist)
fd_keep = keep_cut_sim.calculate('fuel_duty', year_dist)
fd_rpi = rpi_sim.calculate('fuel_duty', year_dist)
decile = baseline_sim.calculate('household_income_decile', year_dist)
net_income = baseline_sim.calculate('household_net_income', year_dist)

# MicroSeries.groupby is weight-aware
avg_base = fd_base.groupby(decile).mean()
avg_keep = fd_keep.groupby(decile).mean()
avg_rpi = fd_rpi.groupby(decile).mean()
avg_income = net_income.groupby(decile).mean()

dist = pd.DataFrame({
    'avg_baseline': avg_base,
    'avg_keep_cut': avg_keep,
    'avg_rpi': avg_rpi,
    'avg_net_income': avg_income,
})
dist.index.name = 'decile'
dist = dist.reset_index()
dist = dist[dist['decile'] >= 1].copy()
dist['benefit_keep_cut'] = dist['avg_baseline'] - dist['avg_keep_cut']
dist['extra_cost_rpi'] = dist['avg_rpi'] - dist['avg_baseline']
dist['benefit_keep_cut_pct_income'] = 100 * dist['benefit_keep_cut'] / dist['avg_net_income']
dist['extra_cost_rpi_pct_income'] = 100 * dist['extra_cost_rpi'] / dist['avg_net_income']
dist.round(2)

In [ ]:
fig = px.bar(
    dist, x='decile', y='benefit_keep_cut_pct_income',
    title=f'Saving from keeping the 5p cut, by decile ({year_dist})',
    labels={'decile': 'Household income decile',
            'benefit_keep_cut_pct_income': '% of household net income'},
    text=dist['benefit_keep_cut_pct_income'].map(lambda v: f"{v:.2f}%"),
)
fig.update_traces(marker_color=PE_BLUE, textposition='outside',
                  textfont=dict(family=PE_FONT, color=PE_BLUE, size=11))
fig.update_layout(
    width=950, height=480, showlegend=False,
    xaxis=dict(tickmode='array', tickvals=list(range(1, 11))),
    yaxis=dict(ticksuffix='%'),
)
fig.show()

## Headline numbers

In [ ]:
print('=' * 72)
print('FUEL DUTY — KEY FIGURES (May 2026)')
print('=' * 72)

print('\n1) Cost of scrapping the planned 5p reversal (Autumn Budget 2025)')
for _, r in scrap_5p.iterrows():
    fy = int(r['Year']); print(f"   {fy}-{(fy+1)%100:02d}:  rate {r['Baseline rate (p/L)']:>5}p   baseline £{r['Baseline revenue (£bn)']:>5.2f}bn   cost £{r['Cost to Treasury (£bn)']:>5.2f}bn")
print(f"   Cumulative {FIRST_DATA_YEAR}-{LAST_DATA_YEAR}: £{scrap_5p['Cost to Treasury (£bn)'].sum():.1f} bn")

print(f'\n2) Revenue forgone vs RPI counterfactual (PE-UK microsim, {FIRST_DATA_YEAR}-{LAST_DATA_YEAR})')
for _, r in rpi_revenue.iterrows():
    fy = int(r['Year']); print(f"   {fy}-{(fy+1)%100:02d}: actual {r['Baseline rate (p/L)']:>6}p  vs counterfactual {r['RPI counterfactual rate (p/L)']:>6}p   uplift £{r['Revenue uplift (£bn)']:>5.1f}bn")
print(f"   Cumulative uplift: £{rpi_revenue['Revenue uplift (£bn)'].sum():.1f} bn over {len(DATA_YEARS)} years (avg £{rpi_revenue['Revenue uplift (£bn)'].mean():.1f}bn/yr)")

print(f'\n3) Rate today (2026) if uprated by RPI since {FIRST_FREEZE_YEAR}')
print(f"   Actual rate:        {actual_rate[2026]*100:6.2f} p/L")
print(f"   RPI counterfactual: {counterfactual_rate[2026]*100:6.2f} p/L")
print(f"   Gap:                {current_gap_p:6.2f} p/L  ({counterfactual_rate[2026]/actual_rate[2026]:.2f}x current)")

print(f'\n4) Distributional impact ({year_dist}, by household income decile)')
print('   Saving if the 5p cut is kept vs the planned reversal:')
for _, r in dist.iterrows():
    print(f"       D{int(r['decile']):>2}:  £{r['benefit_keep_cut']:>5.0f}/yr  ({r['benefit_keep_cut_pct_income']:>4.2f}% of net income)")
print('   Extra cost under the RPI counterfactual:')
for _, r in dist.iterrows():
    print(f"       D{int(r['decile']):>2}:  £{r['extra_cost_rpi']:>5.0f}/yr  ({r['extra_cost_rpi_pct_income']:>4.2f}% of net income)")
print('=' * 72)

### Sources and method

- Household-level revenue and distributional figures are produced by PolicyEngine UK on the latest enhanced Family Resources Survey 2023-29 dataset.
- Historical fuel-duty receipts (2010-11 → 2024-25) are HMRC's published out-turn from the UK Tax & NICs receipts publication on gov.uk.
- The RPI series used to compound the counterfactual is the OBR's March 2026 Economic and Fiscal Outlook.
- No behavioural responses are modelled: fuel volumes are held fixed across scenarios.
- Income deciles use household net income equivalised for household size (the HBAI standard).